## esquema ast

estos son los conceptos clave de programación funcional y cálculo lambda involucrados en el esquema:   

1) Cálculo Lambda Simplemente Tipado (STLC): Se implementan las tres construcciones teóricas fundamentales del modelo: variables (Var), abstracción funcional (Lam, que define una función ligando un parámetro) y aplicación funcional (App, que evalúa una función pasándole un argumento).   

2) Tipos de Datos Algebraicos (ADT): La estructura del AST está concebida como una suma de productos, lo que en lenguajes funcionales como Haskell se modela directamente mediante constructores disjuntos (Var | Literal | Lam | App | BinaryOp | IfThenElse).   

3) Funciones como ciudadanos de primera clase (First-Class Functions): Al soportar tipos flecha (Arrow: from -> to), las funciones pueden ser tratadas como cualquier otro valor primitivo, permitiendo ser pasadas por parámetro o devueltas como resultado de una expresión.

4) Transparencia referencial y evaluación por expresiones: No existen asignaciones destructivas de estado (como variables mutables o sentencias x = x + 1). Toda construcción es una expresión que evalúa de forma determinística a un valor único sin efectos secundarios.   

5) Condicional funcional completo: A diferencia de las estructuras if imperativas, el nodo IfThenElse es una expresión de bifurcación pura: exige obligatoriamente las ramas then y else, garantizando que ambas retornen el mismo tipo de dato para preservar la consistencia matemática.

6) Sistema de tipado estático estricto: El esquema impone la declaración explícita de tipos base y tipos flecha, habilitando la verificación estática (typechecking) y garantizando la preservación y progreso del cálculo antes de cualquier fase de ejecución.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
STLC_AST_Schema = {
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "STLC_AST_Schema",
  "description": "Especificacion JSON Schema para la gramatica STLC: Var, Literal, Lam, App, BinaryOp, IfThenElse",
  "$ref": "#/definitions/Expr",
  "definitions": {
    "Type": {
      "type": "object",
      "oneOf": [
        {
          "description": "Tipos base",
          "properties": {
            "kind": { "type": "string", "enum": ["Base"] },
            "name": { "type": "string", "enum": ["Double", "Int", "Bool", "String"] }
          },
          "required": ["kind", "name"],
          "additionalProperties": False
        },
        {
          "description": "Tipo funcion (T1 -> T2)",
          "properties": {
            "kind": { "type": "string", "enum": ["Arrow"] },
            "from": { "$ref": "#/definitions/Type" },
            "to": { "$ref": "#/definitions/Type" }
          },
          "required": ["kind", "from", "to"],
          "additionalProperties": False
        }
      ]
    },
    "Expr": {
      "type": "object",
      "oneOf": [
        {
          "description": "Var: Referencia a una variable o identificador",
          "properties": {
            "tag": { "type": "string", "enum": ["Var"] },
            "name": { "type": "string" }
          },
          "required": ["tag", "name"],
          "additionalProperties": False
        },
        {
          "description": "Literal: Constante primitiva tipada",
          "properties": {
            "tag": { "type": "string", "enum": ["Literal"] },
            "type": { "$ref": "#/definitions/Type" },
            "value": { "type": ["number", "boolean", "string"] }
          },
          "required": ["tag", "type", "value"],
          "additionalProperties": False
        },
        {
          "description": "Lam: Abstraccion lambda (definicion de funcion)",
          "properties": {
            "tag": { "type": "string", "enum": ["Lam"] },
            "param": { "type": "string" },
            "param_type": { "$ref": "#/definitions/Type" },
            "body": { "$ref": "#/definitions/Expr" }
          },
          "required": ["tag", "param", "param_type", "body"],
          "additionalProperties": False
        },
        {
          "description": "App: Aplicacion de una funcion a un argumento",
          "properties": {
            "tag": { "type": "string", "enum": ["App"] },
            "func": { "$ref": "#/definitions/Expr" },
            "arg": { "$ref": "#/definitions/Expr" }
          },
          "required": ["tag", "func", "arg"],
          "additionalProperties": False
        },
        {
          "description": "BinaryOp: Operacion binaria (aritmetica, comparacion, logica)",
          "properties": {
            "tag": { "type": "string", "enum": ["BinaryOp"] },
            "op": {
              "type": "string",
              "enum": ["+", "-", "*", "/", ">", "<", ">=", "<=", "==", "&&", "||"]
            },
            "left": { "$ref": "#/definitions/Expr" },
            "right": { "$ref": "#/definitions/Expr" }
          },
          "required": ["tag", "op", "left", "right"],
          "additionalProperties": False
        },
        {
          "description": "IfThenElse: Bifurcacion condicional funcional",
          "properties": {
            "tag": { "type": "string", "enum": ["IfThenElse"] },
            "cond": { "$ref": "#/definitions/Expr" },
            "then": { "$ref": "#/definitions/Expr" },
            "else": { "$ref": "#/definitions/Expr" }
          },
          "required": ["tag", "cond", "then", "else"],
          "additionalProperties": False
        }
      ]
    }
  }
}

In [4]:
import json

output_path = '/content/gdrive/MyDrive/1i4St8f2vhpUsPZNuvMRciBeuU7uBvr2a/STLC_AST_Schema.json'

# Asegurarse de que el directorio exista
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(STLC_AST_Schema, f, indent=2)

print(f"Esquema JSON guardado en: {output_path}")

Esquema JSON guardado en: /content/gdrive/MyDrive/1i4St8f2vhpUsPZNuvMRciBeuU7uBvr2a/STLC_AST_Schema.json


## esquema de salida

Propósito:
Define el contrato formal (JSON Schema) que debe cumplir cada línea de  resultados generada en el archivo final 'experimentos.jsonl'.

Utilidad en este benchmark:
1. Estandarización: Asegura que las 3 ramas evaluadas (Baseline 1,  Baseline 2 y la propuesta con DSL/Haskell) guarden exactamente las  mismas métricas (identificador, código generado, validez sintáctica,  validación estática, éxito en runtime, pass@1 y tiempo).

2. Integridad de datos: Permite validar automáticamente cada registro antes de escribirlo, evitando guardar filas corruptas o con campos faltantes.

3. Facilidad de análisis: Garantiza un formato uniforme y predecible para cargar los resultados directamente en Pandas y calcular las métricas finales.

In [5]:
OutputRecordSchema = {
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "OutputRecordSchema",
  "description": "Esquema JSON Schema para registrar el resultado de una prueba en formato JSON Lines (JSONL)",
  "type": "object",
  "properties": {
    "case_id": {
      "type": "string",
      "description": "Identificador unico de la regla o caso de prueba (ej: RULE_001)"
    },
    "lane": {
      "type": "string",
      "enum": ["baseline_1_free_python", "baseline_2_mypy_python", "proposed_dsl_haskell"],
      "description": "Carril experimental evaluado"
    },
    "prompt_input": {
      "type": "string",
      "description": "Texto en lenguaje natural enviado al LLM"
    },
    "raw_output": {
      "type": "string",
      "description": "Respuesta cruda generada por el LLM (codigo Python o string JSON)"
    },
    "syntax_valid": {
      "type": "boolean",
      "description": "Indica si la respuesta cumple con la sintaxis basica requerida"
    },
    "static_validation_passed": {
      "type": "boolean",
      "description": "Indica si supero la auditoria estatica (mypy en carril 2 o typechecker Haskell en carril 3; true por defecto en carril 1)"
    },
    "static_error_message": {
      "type": ["string", "null"],
      "description": "Detalle del error interceptado antes de la ejecucion, si lo hubo"
    },
    "runtime_passed": {
      "type": "boolean",
      "description": "Indica si el codigo o regla se ejecuto sin lanzar excepciones en tiempo de ejecucion"
    },
    "runtime_error_message": {
      "type": ["string", "null"],
      "description": "Mensaje de excepcion ocurrida durante el runtime, si la hubo"
    },
    "test_passed": {
      "type": "boolean",
      "description": "Indica si el resultado calculado coincide con el valor esperado de la regla (pass@1)"
    },
    "execution_time_ms": {
      "type": "number",
      "description": "Tiempo total de procesamiento en milisegundos"
    },
    "timestamp": {
      "type": "string",
      "format": "date-time",
      "description": "Momento exacto en que se registro la prueba"
    }
  },
  "required": [
    "case_id",
    "lane",
    "prompt_input",
    "raw_output",
    "syntax_valid",
    "static_validation_passed",
    "runtime_passed",
    "test_passed",
    "execution_time_ms",
    "timestamp"
  ],
  "additionalProperties": False
}

In [10]:
import json
import os

output_record_path = '/content/gdrive/MyDrive/programacion_funcional/output-record-schema.json'
os.makedirs(os.path.dirname(output_record_path), exist_ok=True)

with open(output_record_path, 'w') as f:
    json.dump(OutputRecordSchema, f, indent=2)

print(f"Esquema de salida guardado exitosamente en: {output_record_path}")

Esquema de salida guardado exitosamente en: /content/gdrive/MyDrive/programacion_funcional/output-record-schema.json
